# 3. Performance Analysis

Calculate sales, profit, margin, product, region, and customer segment metrics used for business decision-making.

## Environment Setup

Load project paths and display settings used across the analysis notebooks.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets" / "screenshots"
DATA_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## Analysis Dataset Load

Import the analysis-ready dataset created during data preparation.

In [ ]:
eda_path = DATA_DIR / "cleaned_data_for_EDA.csv"
df = pd.read_csv(eda_path, parse_dates=["Order Date", "Ship Date"])
df.head()

## KPI Summary

Calculate total sales, total profit, and profit margin for the executive summary.

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = total_profit / total_sales

kpi_summary = pd.DataFrame({
    "KPI": ["Total Sales", "Total Profit", "Profit Margin"],
    "Result": [f"${total_sales:,.2f}", f"${total_profit:,.2f}", f"{profit_margin:.2%}"],
})
kpi_summary

## Category Performance

Compare sales, profit, and margin across major product categories.

In [ ]:
category_summary = (
    df.groupby("Category", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .assign(Profit_Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Profit", ascending=False)
)
category_summary

## Sub-Category Performance

Rank sub-categories by profitability and discount exposure to identify key margin drivers.

In [ ]:
sub_category_summary = (
    df.groupby("Sub-Category", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Average_Discount=("Discount", "mean"))
    .assign(Profit_Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Profit", ascending=False)
)
sub_category_summary.head(10)

## Loss-Making Products

Isolate sub-categories with negative profit for margin improvement opportunities.

In [ ]:
loss_making_subcategories = sub_category_summary[sub_category_summary["Profit"] < 0].sort_values("Profit")
loss_making_subcategories

## Regional Performance

Evaluate sales, profit, discounting, and margin across regions.

In [ ]:
region_summary = (
    df.groupby("Region", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Average_Discount=("Discount", "mean"))
    .assign(Profit_Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Sales", ascending=False)
)
region_summary

## Customer Segment Performance

Measure sales and profit contribution by customer segment.

In [ ]:
segment_summary = (
    df.groupby("Segment", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .assign(Profit_Margin=lambda x: x["Profit"] / x["Sales"])
    .sort_values("Sales", ascending=False)
)
segment_summary

## Visualization Dataset Export

Save the final dataset used by the reporting visuals notebook.

In [ ]:
visualization_path = DATA_DIR / "cleaned_data_for_EDA_visualization.csv"
df.to_csv(visualization_path, index=False)
print(f"Saved visualization dataset to: {visualization_path}")